# Body Measurement Service - Detailed Step-by-Step Analysis

This notebook demonstrates the body measurement extraction process in detail, showing each step from image loading to final measurements.

## Overview
- Image preprocessing
- Pose landmark detection using MediaPipe
- Depth estimation using MiDaS
- Measurement calculations
- Visualization of results

## Step 1: Import Required Libraries

In [ ]:
import sys
import os
from pathlib import Path

# Add parent directory to path to import app modules
sys.path.append(os.path.abspath('..'))

import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle
import mediapipe as mp
import torch
from PIL import Image

# Import our service
from app.services.body_measurement import BodyMeasurementService, PoseLandmark

print("✓ All libraries imported successfully")

## Step 2: Initialize the Body Measurement Service

This will download MediaPipe models if they don't exist and initialize the pose detector and depth estimation model.

In [ ]:
# Initialize service
print("Initializing Body Measurement Service...")
service = BodyMeasurementService()
print("✓ Service initialized successfully")
print(f"✓ Pose landmarker ready")
print(f"✓ Depth estimation model loaded")

## Step 3: Load and Display Test Image

Load a test image of a person in a standing position.

In [ ]:
# Specify your test image path here
test_image_path = "../temp/upload/test_person.jpg"  # Change this to your image path

# Check if file exists
if not os.path.exists(test_image_path):
    print(f"⚠️ Image not found at: {test_image_path}")
    print("Please update the path to point to a valid test image")
else:
    # Load image
    image = cv2.imread(test_image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Display original image
    plt.figure(figsize=(10, 12))
    plt.imshow(image_rgb)
    plt.title("Original Image")
    plt.axis('off')
    plt.show()
    
    print(f"✓ Image loaded: {image.shape[1]}x{image.shape[0]} pixels")

## Step 4: Validate Image (Check if Full Body is Visible)

Before processing, we validate that:
- A person is detected
- Full body is visible (not just face)
- All required landmarks are detected

In [ ]:
# Validate the image
is_valid, message = service.validate_front_image(image)

print(f"Validation Result: {'✓ PASS' if is_valid else '✗ FAIL'}")
print(f"Message: {message}")

if not is_valid:
    print("\n⚠️ Image validation failed. Please use a different image.")

## Step 5: Detect Pose Landmarks

Use MediaPipe to detect 33 body landmarks including shoulders, hips, knees, etc.

In [ ]:
# Convert image to MediaPipe format
rgb_frame = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

# Detect pose
detection_result = service.pose_landmarker.detect(mp_image)

if detection_result.pose_landmarks:
    landmarks = detection_result.pose_landmarks[0]
    print(f"✓ Detected {len(landmarks)} pose landmarks")
    
    # Display key landmarks
    key_landmarks = [
        (PoseLandmark.NOSE, "Nose"),
        (PoseLandmark.LEFT_SHOULDER, "Left Shoulder"),
        (PoseLandmark.RIGHT_SHOULDER, "Right Shoulder"),
        (PoseLandmark.LEFT_HIP, "Left Hip"),
        (PoseLandmark.RIGHT_HIP, "Right Hip"),
        (PoseLandmark.LEFT_KNEE, "Left Knee"),
        (PoseLandmark.RIGHT_KNEE, "Right Knee"),
        (PoseLandmark.LEFT_ANKLE, "Left Ankle"),
        (PoseLandmark.RIGHT_ANKLE, "Right Ankle"),
    ]
    
    print("\nKey Landmark Positions (normalized x, y):")
    for idx, name in key_landmarks:
        lm = landmarks[idx]
        print(f"  {name:20s}: ({lm.x:.3f}, {lm.y:.3f}) - visibility: {lm.visibility:.2f}")
else:
    print("✗ No pose landmarks detected")

## Step 6: Visualize Pose Landmarks on Image

Draw the detected landmarks and connections on the image.

In [ ]:
# Create a copy for visualization
annotated_image = image_rgb.copy()
height, width, _ = annotated_image.shape

# Draw landmarks
if detection_result.pose_landmarks:
    mp_drawing = mp.solutions.drawing_utils
    mp_pose = mp.solutions.pose
    
    # Convert landmarks to the format expected by drawing_utils
    landmark_list = detection_result.pose_landmarks[0]
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 14))
    ax.imshow(annotated_image)
    
    # Draw landmarks as circles
    for idx, landmark in enumerate(landmark_list):
        x = int(landmark.x * width)
        y = int(landmark.y * height)
        
        # Different colors for different body parts
        if idx in [PoseLandmark.LEFT_SHOULDER, PoseLandmark.RIGHT_SHOULDER]:
            color = 'red'
            size = 100
        elif idx in [PoseLandmark.LEFT_HIP, PoseLandmark.RIGHT_HIP]:
            color = 'blue'
            size = 100
        elif idx in [PoseLandmark.LEFT_KNEE, PoseLandmark.RIGHT_KNEE]:
            color = 'green'
            size = 80
        else:
            color = 'yellow'
            size = 50
        
        ax.scatter(x, y, c=color, s=size, alpha=0.7, edgecolors='black', linewidths=2)
    
    # Draw connections for key measurements
    def draw_line(idx1, idx2, color, linewidth=2, label=None):
        lm1 = landmark_list[idx1]
        lm2 = landmark_list[idx2]
        x1, y1 = int(lm1.x * width), int(lm1.y * height)
        x2, y2 = int(lm2.x * width), int(lm2.y * height)
        ax.plot([x1, x2], [y1, y2], color=color, linewidth=linewidth, label=label)
    
    # Draw key measurement lines
    draw_line(PoseLandmark.LEFT_SHOULDER, PoseLandmark.RIGHT_SHOULDER, 'red', 3, 'Shoulders')
    draw_line(PoseLandmark.LEFT_HIP, PoseLandmark.RIGHT_HIP, 'blue', 3, 'Hips')
    draw_line(PoseLandmark.LEFT_SHOULDER, PoseLandmark.LEFT_HIP, 'orange', 2, 'Torso Left')
    draw_line(PoseLandmark.RIGHT_SHOULDER, PoseLandmark.RIGHT_HIP, 'orange', 2, 'Torso Right')
    draw_line(PoseLandmark.LEFT_HIP, PoseLandmark.LEFT_KNEE, 'green', 2)
    draw_line(PoseLandmark.RIGHT_HIP, PoseLandmark.RIGHT_KNEE, 'green', 2)
    draw_line(PoseLandmark.LEFT_KNEE, PoseLandmark.LEFT_ANKLE, 'purple', 2)
    draw_line(PoseLandmark.RIGHT_KNEE, PoseLandmark.RIGHT_ANKLE, 'purple', 2)
    
    ax.set_title("Detected Pose Landmarks", fontsize=16, fontweight='bold')
    ax.axis('off')
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.show()
    
    print("✓ Landmarks visualized")

## Step 7: Calculate Scale Factor from User Height

To convert pixel measurements to centimeters, we need a scale factor.
We calculate this using the user's known height.

In [ ]:
# Set user height (in cm)
user_height_cm = 170.0  # Change this to actual user height

# Calculate scale factor
image_height = image.shape[0]
distance, scale_factor = service.calculate_distance_using_height(
    landmarks, image_height, user_height_cm
)

print(f"User Height: {user_height_cm} cm")
print(f"Image Height: {image_height} pixels")
print(f"Calculated Distance: {distance:.2f} units")
print(f"Scale Factor: {scale_factor:.4f} cm/pixel")

# Calculate person height in pixels
nose = landmarks[PoseLandmark.NOSE]
left_ankle = landmarks[PoseLandmark.LEFT_ANKLE]
right_ankle = landmarks[PoseLandmark.RIGHT_ANKLE]

top_head = nose.y * image_height
bottom_foot = max(left_ankle.y, right_ankle.y) * image_height
person_height_px = abs(bottom_foot - top_head)

print(f"\nPerson Height in Image: {person_height_px:.1f} pixels")
print(f"Verification: {person_height_px * scale_factor:.1f} cm (should be ≈ {user_height_cm} cm)")

## Step 8: Estimate Depth Map

Use MiDaS depth estimation model to understand the 3D structure of the body.
This helps improve circumference calculations.

In [ ]:
print("Estimating depth map...")
depth_map = service.estimate_depth(image)
print(f"✓ Depth map generated: {depth_map.shape}")

# Visualize depth map
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Original image
axes[0].imshow(image_rgb)
axes[0].set_title("Original Image", fontsize=14, fontweight='bold')
axes[0].axis('off')

# Depth map
depth_normalized = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min())
im = axes[1].imshow(depth_normalized, cmap='plasma')
axes[1].set_title("Depth Map (Warmer = Closer)", fontsize=14, fontweight='bold')
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print(f"Depth range: {depth_map.min():.2f} to {depth_map.max():.2f}")

## Step 9: Calculate All Body Measurements

Now we calculate all measurements including:
- Shoulder width
- Chest/bust circumference
- Waist
- Hip circumference
- Arm length
- Thigh circumference
- And more...

In [ ]:
# Create a result object compatible with calculate_measurements
class PoseResult:
    def __init__(self, landmarks):
        self.pose_landmarks = type('obj', (object,), {'landmark': landmarks})()

results = PoseResult(landmarks)

# Calculate measurements
image_width = image.shape[1]
measurements = service.calculate_measurements(
    results,
    scale_factor,
    image_width,
    image_height,
    depth_map,
    image,
    user_height_cm
)

print("\n" + "="*50)
print("BODY MEASUREMENTS (in cm)")
print("="*50)

# Group measurements by category
upper_body = {
    'Shoulder Width': measurements.get('shoulder_width'),
    'Chest Width': measurements.get('chest_width'),
    'Chest Circumference': measurements.get('chest_circumference'),
    'Waist': measurements.get('waist'),
    'Neck': measurements.get('neck'),
    'Arm Length': measurements.get('arm_length'),
    'Shirt Length': measurements.get('shirt_length'),
}

lower_body = {
    'Hip Width': measurements.get('hip_width'),
    'Hip Circumference': measurements.get('hip'),
    'Thigh': measurements.get('thigh'),
    'Thigh Circumference': measurements.get('thigh_circumference'),
    'Trouser Length': measurements.get('trouser_length'),
}

print("\nUpper Body:")
for name, value in upper_body.items():
    if value:
        print(f"  {name:25s}: {value:6.2f} cm")

print("\nLower Body:")
for name, value in lower_body.items():
    if value:
        print(f"  {name:25s}: {value:6.2f} cm")

print("\n" + "="*50)

## Step 10: Visualize Measurement Points

Show where each measurement is taken on the body.

In [ ]:
# Create visualization with measurement annotations
fig, ax = plt.subplots(figsize=(12, 16))
ax.imshow(image_rgb)

# Get key landmarks
left_shoulder = landmarks[PoseLandmark.LEFT_SHOULDER]
right_shoulder = landmarks[PoseLandmark.RIGHT_SHOULDER]
left_hip = landmarks[PoseLandmark.LEFT_HIP]
right_hip = landmarks[PoseLandmark.RIGHT_HIP]
left_knee = landmarks[PoseLandmark.LEFT_KNEE]
nose = landmarks[PoseLandmark.NOSE]

# Helper function to draw measurement lines
def draw_measurement(y_pos, label, color='yellow'):
    y_px = int(y_pos * height)
    ax.axhline(y=y_px, color=color, linestyle='--', linewidth=2, alpha=0.7)
    ax.text(width * 0.05, y_px, label, color=color, fontsize=10, 
            fontweight='bold', bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

# Draw measurement lines
chest_y = left_shoulder.y + (left_hip.y - left_shoulder.y) * 0.15
waist_y = left_shoulder.y + (left_hip.y - left_shoulder.y) * 0.35
hip_y = left_hip.y
thigh_y = left_hip.y + (left_knee.y - left_hip.y) * 0.2

draw_measurement(left_shoulder.y, f"Shoulders: {measurements['shoulder_width']:.1f} cm", 'red')
draw_measurement(chest_y, f"Chest: {measurements['chest_circumference']:.1f} cm", 'orange')
draw_measurement(waist_y, f"Waist: {measurements['waist']:.1f} cm", 'yellow')
draw_measurement(hip_y, f"Hips: {measurements['hip']:.1f} cm", 'blue')
draw_measurement(thigh_y, f"Thigh: {measurements['thigh_circumference']:.1f} cm", 'green')

ax.set_title("Body Measurements Visualization", fontsize=16, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

## Step 11: Create Measurement Summary Chart

In [ ]:
# Create bar chart of measurements
circumference_measurements = {
    'Neck': measurements.get('neck', 0),
    'Chest': measurements.get('chest_circumference', 0),
    'Waist': measurements.get('waist', 0),
    'Hip': measurements.get('hip', 0),
    'Thigh': measurements.get('thigh_circumference', 0),
}

length_measurements = {
    'Shoulder Width': measurements.get('shoulder_width', 0),
    'Arm Length': measurements.get('arm_length', 0),
    'Shirt Length': measurements.get('shirt_length', 0),
    'Trouser Length': measurements.get('trouser_length', 0),
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Circumference measurements
colors1 = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8']
bars1 = ax1.bar(circumference_measurements.keys(), circumference_measurements.values(), color=colors1)
ax1.set_title('Circumference Measurements', fontsize=14, fontweight='bold')
ax1.set_ylabel('Centimeters (cm)', fontsize=12)
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim(0, max(circumference_measurements.values()) * 1.2)

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}', ha='center', va='bottom', fontweight='bold')

# Length measurements
colors2 = ['#95E1D3', '#F38181', '#AA96DA', '#FCBAD3']
bars2 = ax2.bar(length_measurements.keys(), length_measurements.values(), color=colors2)
ax2.set_title('Length Measurements', fontsize=14, fontweight='bold')
ax2.set_ylabel('Centimeters (cm)', fontsize=12)
ax2.grid(axis='y', alpha=0.3)
ax2.set_ylim(0, max(length_measurements.values()) * 1.2)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=15, ha='right')

# Add value labels on bars
for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## Step 12: Export Measurements to Dictionary/JSON

In [ ]:
import json

# Prepare complete measurement report
measurement_report = {
    "user_height_cm": user_height_cm,
    "scale_factor": scale_factor,
    "measurements_cm": measurements,
    "measurement_categories": {
        "upper_body": upper_body,
        "lower_body": lower_body
    }
}

# Pretty print JSON
print(json.dumps(measurement_report, indent=2))

# Optionally save to file
output_path = "../temp/output/measurement_report.json"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, 'w') as f:
    json.dump(measurement_report, f, indent=2)
print(f"\n✓ Measurement report saved to: {output_path}")

## Step 13: Test with API (Optional)

Test the full API endpoint if the server is running.

In [ ]:
import requests

# API endpoint URL
API_URL = "http://localhost:8000/api/body-measurement/measure"

# Read image file
try:
    with open(test_image_path, 'rb') as f:
        files = {'front_image': f}
        data = {'height': user_height_cm}
        
        print(f"Sending request to API: {API_URL}")
        response = requests.post(API_URL, files=files, data=data)
        
        if response.status_code == 200:
            result = response.json()
            print("\n✓ API Response:")
            print(json.dumps(result, indent=2))
        else:
            print(f"\n✗ API Error: {response.status_code}")
            print(response.text)
except requests.exceptions.ConnectionError:
    print("⚠️ Could not connect to API. Make sure the server is running.")
    print("Start the server with: uvicorn app.main:app --reload")
except FileNotFoundError:
    print("⚠️ Test image not found. Please provide a valid image path.")

## Summary

This notebook demonstrated the complete body measurement process:

1. ✓ Image loading and validation
2. ✓ Pose landmark detection (33 points)
3. ✓ Scale factor calculation from known height
4. ✓ Depth estimation for 3D understanding
5. ✓ Body width detection at key points
6. ✓ Measurement calculations (circumferences and lengths)
7. ✓ Visualization of results
8. ✓ Export to JSON format

### Key Measurements Extracted:
- **Upper Body**: Shoulder width, chest, waist, neck, arm length
- **Lower Body**: Hip, thigh, trouser length

### Technologies Used:
- **MediaPipe**: Pose landmark detection
- **MiDaS**: Depth estimation
- **OpenCV**: Image processing
- **NumPy**: Mathematical calculations